# Nonlinear discovery

This notebook is part of the staged manuscript-reproduction workflow. During automated validation it resolves placeholder paths to the deterministic demo dataset; once local real-data paths are configured it will run against the real case-study artifacts.

In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    env_root = os.environ.get("BSM_PUBLIC_RF_REPO_ROOT")
    if env_root:
        candidate = Path(env_root).resolve()
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError("Could not locate the bsm-public-rf repository root.")


REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from bsm_rfm import build_manuscript_notebook_context, run_nonlinear_discovery_stage

context = build_manuscript_notebook_context(REPO_ROOT, "05_nonlinear_discovery.ipynb")
result = run_nonlinear_discovery_stage(context)
result.nonlinear.summary

### Step 1: Generate Nonlinear Bases

Create nonlinear transformations of selected features using splines, logs, and polynomials.

**Basis functions:**
- Splines: cubic with knots at quantiles (3-5 knots per feature)
- Log transformation: log(x + 1) for non-negative features
- Sqrt transformation: sqrt(x) for non-negative features
- Polynomial: x², x³, etc. (separate from Stage 04 polynomial interactions)

**Process:**
1. For top-N features, evaluate each basis function
2. Combine bases into design matrix
3. Remove collinear bases

**Parameters:**
- `basis_functions`: 'spline', 'poly', 'log', 'sqrt', or combined (default 'spline')
- `knot_placement`: 'uniform', 'quantile', or 'manual' (default 'quantile')
- `degrees_of_freedom`: Total DoF budget (default 50)

**Output:**
- `nonlinear_features_df`: Basis function design matrix
- Typically adds 200-400 new features

In [ ]:
result.artifact_paths

### Step 2: Combine All Features

Merge original + interaction + nonlinear features into full matrix.

**Output:**
- Full feature matrix: original + interactions + nonlinear (~400-1100 features)